# 07_03B — Encoders Avançados + Feature-Engine — Early-Stage

## Objetivo

Comparar o baseline corrigido da etapa `07_03A` com pipelines que usam `category_encoders` e `feature-engine`.

## Target

Neste projeto:

```text
target_banco_ganhou = 0 → banco ganhou / êxito
target_banco_ganhou = 1 → banco perdeu / condenação
```

Portanto:

```text
proba_perda = predict_proba(...)[1]
score_risco_perda = proba_perda
score_prioridade_financeira = proba_perda × fe_valor_ajuizado
```

## Modelos testados

- Logistic Regression + CountEncoder
- Logistic Regression + CatBoostEncoder
- Logistic Regression + JamesSteinEncoder
- Logistic Regression + MEstimateEncoder
- Random Forest + encoders supervisionados
- HistGradientBoosting + encoders supervisionados

## Leakage

Todos os encoders supervisionados ficam dentro do `Pipeline`, então cada fold faz `fit` apenas no treino e `transform` na validação.

## Baseline a bater da etapa 3A

```text
PR-AUC perda holdout ≈ 0,4668
ROC-AUC perda holdout ≈ 0,7786
Top 5% risco: precision perda ≈ 60,3%
Top 10% financeiro: captura ≈ 50,3% do valor das perdas
```

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
import importlib.util

warnings.filterwarnings("ignore")

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
)

pd.set_option("display.max_columns", 400)
pd.set_option("display.max_rows", 300)
pd.set_option("display.width", 260)

BASE_DIR = Path("..")
PROCESSED_DIR = BASE_DIR / "data" / "processed"
REPORTS_DIR = BASE_DIR / "outputs" / "reports" / "modelagem_early_stage"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

DEV_HIST_FILE = PROCESSED_DIR / "abt_early_stage_v1_dev_hist.parquet"
HOLDOUT_HIST_FILE = PROCESSED_DIR / "abt_early_stage_v1_holdout_hist.parquet"
FEATURE_LIST_HIST_FILE = PROCESSED_DIR / "feature_list_early_stage_v1_hist.txt"

TARGET_COL_LEGACY = "target_banco_ganhou"
TARGET_COL = "target_banco_perdeu"
DATE_COL = "Datainicial"
VALUE_COL = "fe_valor_ajuizado"
RANDOM_STATE = 42

print("Setup concluído.")
print("Target: 0=banco_ganhou | 1=banco_perdeu")

## 2. Checar dependências

In [ ]:
required_packages = {
    "category_encoders": "category_encoders",
    "feature_engine": "feature_engine",
}

dependency_status = []
for import_name, package_name in required_packages.items():
    ok = importlib.util.find_spec(import_name) is not None
    dependency_status.append({
        "import_name": import_name,
        "package_name": package_name,
        "available": ok,
        "install_command": f"pip install {package_name}",
    })

dependency_status = pd.DataFrame(dependency_status)
display(dependency_status)

missing = dependency_status.loc[dependency_status["available"] == False, "package_name"].tolist()
if missing:
    print("Pacotes faltantes:")
    for p in missing:
        print(f"  pip install {p}")
    raise ImportError("Instale os pacotes faltantes antes de rodar este notebook.")

import category_encoders as ce
from feature_engine.encoding import RareLabelEncoder
from feature_engine.selection import DropCorrelatedFeatures

print("Dependências carregadas com sucesso.")

## 3. Carregar bases

In [ ]:
df_dev = pd.read_parquet(DEV_HIST_FILE)
df_holdout = pd.read_parquet(HOLDOUT_HIST_FILE)

with open(FEATURE_LIST_HIST_FILE, "r", encoding="utf-8") as f:
    feature_list = [line.strip() for line in f if line.strip()]

df_dev[DATE_COL] = pd.to_datetime(df_dev[DATE_COL], errors="coerce")
df_holdout[DATE_COL] = pd.to_datetime(df_holdout[DATE_COL], errors="coerce")

df_dev[TARGET_COL] = df_dev[TARGET_COL_LEGACY].astype(int)
df_holdout[TARGET_COL] = df_holdout[TARGET_COL_LEGACY].astype(int)

feature_list = [c for c in feature_list if c in df_dev.columns and c in df_holdout.columns]

print("Dev:", df_dev.shape)
print("Holdout:", df_holdout.shape)
print("Features:", len(feature_list))
df_dev.head()

## 4. Validar distribuição do target

In [ ]:
def target_distribution(df, name):
    out = (
        df[TARGET_COL]
        .value_counts(dropna=False)
        .rename_axis("target")
        .reset_index(name="qtd")
    )
    out["classe"] = out["target"].map({0: "banco_ganhou", 1: "banco_perdeu"})
    out["perc"] = out["qtd"] / out["qtd"].sum()
    out["dataset"] = name
    return out

target_dist = pd.concat([
    target_distribution(df_dev, "dev"),
    target_distribution(df_holdout, "holdout"),
], ignore_index=True)

target_dist.to_csv(REPORTS_DIR / "33_3b_target_distribution.csv", index=False)
target_dist

## 5. Funções auxiliares

In [ ]:
def save_report(df_report, filename):
    path = REPORTS_DIR / filename
    df_report.to_csv(path, index=False)
    print(f"Relatório salvo em: {path}")


def infer_feature_types(df, features):
    numeric_features = []
    categorical_features = []
    text_features = []
    for col in features:
        if col == "fe_texto_inicial_curto":
            text_features.append(col)
        elif pd.api.types.is_numeric_dtype(df[col]):
            numeric_features.append(col)
        else:
            categorical_features.append(col)
    return numeric_features, categorical_features, text_features


def make_temporal_folds(df, date_col=DATE_COL):
    folds = [(0.55, 0.70), (0.70, 0.85), (0.85, 1.00)]
    df_sorted = df.sort_values(date_col).copy()
    dates = df_sorted[date_col]
    split_rows = []
    for i, (train_end_q, val_end_q) in enumerate(folds, start=1):
        train_end_date = dates.quantile(train_end_q)
        val_end_date = dates.quantile(val_end_q)
        train_idx = df_sorted.index[df_sorted[date_col] <= train_end_date]
        val_idx = df_sorted.index[(df_sorted[date_col] > train_end_date) & (df_sorted[date_col] <= val_end_date)]
        split_rows.append({
            "fold": i,
            "train_start_date": df_sorted.loc[train_idx, date_col].min(),
            "train_end_date": train_end_date,
            "val_start_date": df_sorted.loc[val_idx, date_col].min(),
            "val_end_date": val_end_date,
            "train_idx": train_idx,
            "val_idx": val_idx,
            "qtd_train": len(train_idx),
            "qtd_val": len(val_idx),
        })
    return split_rows


def threshold_metrics_perda(y_true, proba_perda, threshold=0.5):
    pred = (proba_perda >= threshold).astype(int)
    return {
        "threshold": threshold,
        "precision_perda": precision_score(y_true, pred, zero_division=0),
        "recall_perda": recall_score(y_true, pred, zero_division=0),
        "f1_perda": f1_score(y_true, pred, zero_division=0),
        "pred_perda_rate": pred.mean(),
    }


def find_best_f1_threshold_perda(y_true, proba_perda):
    precision, recall, thresholds = precision_recall_curve(y_true, proba_perda)
    if len(thresholds) == 0:
        return 0.5, 0, 0, 0
    precision_t = precision[:-1]
    recall_t = recall[:-1]
    f1_t = 2 * precision_t * recall_t / np.maximum(precision_t + recall_t, 1e-12)
    best_idx = np.nanargmax(f1_t)
    return thresholds[best_idx], precision_t[best_idx], recall_t[best_idx], f1_t[best_idx]


def topk_risco_perda_metrics(y_true, proba_perda, ks=(0.01, 0.05, 0.10, 0.20)):
    df_tmp = pd.DataFrame({"y_true": y_true, "score_risco_perda": proba_perda})
    df_tmp = df_tmp.sort_values("score_risco_perda", ascending=False).reset_index(drop=True)
    total_perdas = (df_tmp["y_true"] == 1).sum()
    taxa_perda_base = (df_tmp["y_true"] == 1).mean()
    out = []
    for k in ks:
        n = max(1, int(np.ceil(len(df_tmp) * k)))
        top = df_tmp.head(n)
        precision_perda_at_k = (top["y_true"] == 1).mean()
        recall_perda_at_k = (top["y_true"] == 1).sum() / total_perdas if total_perdas else np.nan
        lift_perda_at_k = precision_perda_at_k / taxa_perda_base if taxa_perda_base else np.nan
        out.append({
            "top_k": k,
            "n_top": n,
            "precision_perda_at_k": precision_perda_at_k,
            "recall_perda_at_k": recall_perda_at_k,
            "lift_perda_at_k": lift_perda_at_k,
            "taxa_perda_base": taxa_perda_base,
        })
    return pd.DataFrame(out)


def topk_prioridade_financeira_metrics(y_true, proba_perda, valor_ajuizado, ks=(0.01, 0.05, 0.10, 0.20)):
    df_tmp = pd.DataFrame({
        "y_true": y_true,
        "proba_perda": proba_perda,
        "valor_ajuizado": pd.to_numeric(valor_ajuizado, errors="coerce").fillna(0),
    })
    df_tmp["score_prioridade_financeira"] = df_tmp["proba_perda"] * df_tmp["valor_ajuizado"]
    df_tmp = df_tmp.sort_values("score_prioridade_financeira", ascending=False).reset_index(drop=True)
    total_perdas = (df_tmp["y_true"] == 1).sum()
    taxa_perda_base = (df_tmp["y_true"] == 1).mean()
    total_valor = df_tmp["valor_ajuizado"].sum()
    total_valor_perdas = df_tmp.loc[df_tmp["y_true"] == 1, "valor_ajuizado"].sum()
    out = []
    for k in ks:
        n = max(1, int(np.ceil(len(df_tmp) * k)))
        top = df_tmp.head(n)
        valor_top = top["valor_ajuizado"].sum()
        valor_perdas_top = top.loc[top["y_true"] == 1, "valor_ajuizado"].sum()
        precision_perda_at_k = (top["y_true"] == 1).mean()
        recall_perda_at_k = (top["y_true"] == 1).sum() / total_perdas if total_perdas else np.nan
        lift_perda_at_k = precision_perda_at_k / taxa_perda_base if taxa_perda_base else np.nan
        out.append({
            "top_k": k,
            "n_top": n,
            "precision_perda_at_k": precision_perda_at_k,
            "recall_perda_at_k": recall_perda_at_k,
            "lift_perda_at_k": lift_perda_at_k,
            "taxa_perda_base": taxa_perda_base,
            "valor_ajuizado_top": valor_top,
            "share_valor_ajuizado_total": valor_top / total_valor if total_valor else np.nan,
            "valor_ajuizado_perdas_top": valor_perdas_top,
            "share_valor_perdas_total": valor_perdas_top / total_valor_perdas if total_valor_perdas else np.nan,
        })
    return pd.DataFrame(out)

## 6. Separar features por tipo

In [ ]:
numeric_features, categorical_features, text_features = infer_feature_types(df_dev, feature_list)

feature_type_summary = pd.DataFrame([
    {"tipo": "numeric", "qtd": len(numeric_features)},
    {"tipo": "categorical", "qtd": len(categorical_features)},
    {"tipo": "text", "qtd": len(text_features)},
])

feature_type_summary

## 7. Cardinalidade das categóricas

In [ ]:
cat_cardinality = (
    pd.DataFrame({
        "feature": categorical_features,
        "nunique_dev": [df_dev[c].nunique(dropna=True) for c in categorical_features],
        "missing_rate_dev": [df_dev[c].isna().mean() for c in categorical_features],
    })
    .sort_values("nunique_dev", ascending=False)
)

save_report(cat_cardinality, "34_3b_categorical_cardinality.csv")
display(cat_cardinality.head(50))

LOW_CARD_MAX_UNIQUE = 30
low_card_cats = cat_cardinality.loc[cat_cardinality["nunique_dev"] <= LOW_CARD_MAX_UNIQUE, "feature"].tolist()
high_card_cats = cat_cardinality.loc[cat_cardinality["nunique_dev"] > LOW_CARD_MAX_UNIQUE, "feature"].tolist()

print("Low-card cats:", len(low_card_cats))
print("High-card cats:", len(high_card_cats))

## 8. Criar folds temporais

In [ ]:
folds = make_temporal_folds(df_dev, DATE_COL)

fold_summary_rows = []
for fold in folds:
    train_idx = fold["train_idx"]
    val_idx = fold["val_idx"]
    taxa_perda_train = df_dev.loc[train_idx, TARGET_COL].mean()
    taxa_perda_val = df_dev.loc[val_idx, TARGET_COL].mean()
    fold_summary_rows.append({
        "fold": fold["fold"],
        "train_start_date": fold["train_start_date"],
        "train_end_date": fold["train_end_date"],
        "val_start_date": fold["val_start_date"],
        "val_end_date": fold["val_end_date"],
        "qtd_train": fold["qtd_train"],
        "qtd_val": fold["qtd_val"],
        "taxa_perda_train": taxa_perda_train,
        "taxa_perda_val": taxa_perda_val,
        "taxa_ganho_train": 1 - taxa_perda_train,
        "taxa_ganho_val": 1 - taxa_perda_val,
    })

fold_summary = pd.DataFrame(fold_summary_rows)
save_report(fold_summary, "35_3b_walk_forward_folds_summary.csv")
fold_summary

## 9. Preprocessadores avançados

In [ ]:
def make_text_pipeline(text_features, tfidf_max_features=1000, tfidf_min_df=30):
    if not text_features:
        return None
    text_col = text_features[0]
    return Pipeline([
        ("selector", FunctionTransformer(lambda x: x[text_col].fillna("").astype(str), validate=False)),
        ("tfidf", TfidfVectorizer(max_features=tfidf_max_features, ngram_range=(1, 2), min_df=tfidf_min_df)),
    ])


def make_numeric_pipeline(scale=True):
    steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale:
        steps.append(("scaler", StandardScaler()))
    return Pipeline(steps)


def make_cat_encoder_pipeline(encoder_name):
    if encoder_name == "count":
        encoder = ce.CountEncoder(handle_unknown=0, handle_missing=0, normalize=False)
    elif encoder_name == "catboost":
        encoder = ce.CatBoostEncoder(handle_unknown="value", handle_missing="value", random_state=RANDOM_STATE, sigma=0.05, a=1.0)
    elif encoder_name == "james_stein":
        encoder = ce.JamesSteinEncoder(handle_unknown="value", handle_missing="value")
    elif encoder_name == "mestimate":
        encoder = ce.MEstimateEncoder(handle_unknown="value", handle_missing="value", m=10.0)
    else:
        raise ValueError(f"Encoder não suportado: {encoder_name}")

    return Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="nao_informado")),
        ("rare", RareLabelEncoder(tol=0.01, n_categories=5, replace_with="categoria_rara")),
        ("encoder", encoder),
    ])


def make_advanced_preprocessor(encoder_name, scale_numeric=True, include_text=True):
    transformers = []
    if numeric_features:
        transformers.append(("num", make_numeric_pipeline(scale=scale_numeric), numeric_features))
    if categorical_features:
        transformers.append(("cat", make_cat_encoder_pipeline(encoder_name), categorical_features))
    if include_text and text_features:
        transformers.append(("txt", make_text_pipeline(text_features), text_features))

    return ColumnTransformer(transformers=transformers, remainder="drop", sparse_threshold=0.3)

## 10. Definir pipelines candidatos

In [ ]:
def make_lr_encoder_pipeline(encoder_name):
    return Pipeline([
        ("preprocessor", make_advanced_preprocessor(encoder_name=encoder_name, scale_numeric=True, include_text=True)),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced", solver="saga", n_jobs=-1, random_state=RANDOM_STATE)),
    ])


def make_rf_encoder_pipeline(encoder_name):
    return Pipeline([
        ("preprocessor", make_advanced_preprocessor(encoder_name=encoder_name, scale_numeric=False, include_text=False)),
        ("drop_corr", DropCorrelatedFeatures(threshold=0.95)),
        ("model", RandomForestClassifier(
            n_estimators=400,
            max_depth=18,
            min_samples_split=30,
            min_samples_leaf=15,
            max_features="sqrt",
            max_samples=0.8,
            class_weight="balanced",
            criterion="entropy",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )),
    ])


def make_hgb_encoder_pipeline(encoder_name):
    return Pipeline([
        ("preprocessor", make_advanced_preprocessor(encoder_name=encoder_name, scale_numeric=False, include_text=False)),
        ("drop_corr", DropCorrelatedFeatures(threshold=0.95)),
        ("model", HistGradientBoostingClassifier(
            learning_rate=0.05,
            max_iter=300,
            max_leaf_nodes=31,
            l2_regularization=1.0,
            random_state=RANDOM_STATE,
        )),
    ])


model_factories = {
    "lr_count_encoder_tfidf": lambda: make_lr_encoder_pipeline("count"),
    "lr_catboost_encoder_tfidf": lambda: make_lr_encoder_pipeline("catboost"),
    "lr_james_stein_encoder_tfidf": lambda: make_lr_encoder_pipeline("james_stein"),
    "lr_mestimate_encoder_tfidf": lambda: make_lr_encoder_pipeline("mestimate"),

    "rf_count_encoder": lambda: make_rf_encoder_pipeline("count"),
    "rf_catboost_encoder": lambda: make_rf_encoder_pipeline("catboost"),
    "rf_james_stein_encoder": lambda: make_rf_encoder_pipeline("james_stein"),
    "rf_mestimate_encoder": lambda: make_rf_encoder_pipeline("mestimate"),

    "hgb_count_encoder": lambda: make_hgb_encoder_pipeline("count"),
    "hgb_catboost_encoder": lambda: make_hgb_encoder_pipeline("catboost"),
    "hgb_james_stein_encoder": lambda: make_hgb_encoder_pipeline("james_stein"),
    "hgb_mestimate_encoder": lambda: make_hgb_encoder_pipeline("mestimate"),
}

pd.DataFrame({"pipeline": list(model_factories.keys())})

## 11. Configuração rápida vs completa

In [ ]:
RUN_MODE = "fast"  # altere para "full" se quiser testar todos

if RUN_MODE == "fast":
    selected_models = {
        "lr_catboost_encoder_tfidf": model_factories["lr_catboost_encoder_tfidf"],
        "lr_james_stein_encoder_tfidf": model_factories["lr_james_stein_encoder_tfidf"],
        "hgb_catboost_encoder": model_factories["hgb_catboost_encoder"],
        "hgb_mestimate_encoder": model_factories["hgb_mestimate_encoder"],
    }
else:
    selected_models = model_factories

print("Pipelines selecionados:")
for k in selected_models:
    print(" -", k)

## 12. Rodar walk-forward

In [ ]:
def evaluate_model_on_fold(model_name, pipeline, df, train_idx, val_idx, features):
    X_train = df.loc[train_idx, features].copy()
    y_train = df.loc[train_idx, TARGET_COL].copy()
    X_val = df.loc[val_idx, features].copy()
    y_val = df.loc[val_idx, TARGET_COL].copy()

    pipeline.fit(X_train, y_train)
    proba_perda_val = pipeline.predict_proba(X_val)[:, 1]

    roc_auc_perda = roc_auc_score(y_val, proba_perda_val)
    pr_auc_perda = average_precision_score(y_val, proba_perda_val)
    best_thr, best_p, best_r, best_f1 = find_best_f1_threshold_perda(y_val, proba_perda_val)
    thr_05 = threshold_metrics_perda(y_val, proba_perda_val, threshold=0.5)

    result = {
        "model": model_name,
        "roc_auc_perda": roc_auc_perda,
        "pr_auc_perda": pr_auc_perda,
        "taxa_perda_val": y_val.mean(),
        "taxa_ganho_val": 1 - y_val.mean(),
        "best_f1_threshold_perda": best_thr,
        "best_f1_precision_perda": best_p,
        "best_f1_recall_perda": best_r,
        "best_f1_perda": best_f1,
        "precision_perda_t05": thr_05["precision_perda"],
        "recall_perda_t05": thr_05["recall_perda"],
        "f1_perda_t05": thr_05["f1_perda"],
        "pred_perda_rate_t05": thr_05["pred_perda_rate"],
    }

    topk_perda = topk_risco_perda_metrics(y_val.values, proba_perda_val)

    if VALUE_COL in df.columns:
        topk_financeiro = topk_prioridade_financeira_metrics(
            y_true=y_val.values,
            proba_perda=proba_perda_val,
            valor_ajuizado=df.loc[val_idx, VALUE_COL].values
        )
    else:
        topk_financeiro = pd.DataFrame()

    return result, topk_perda, topk_financeiro


walk_forward_results = []
topk_perda_results = []
topk_financeiro_results = []

for model_name, factory in selected_models.items():
    print("=" * 100)
    print(f"Modelo: {model_name}")
    for fold in folds:
        print(f"  Fold {fold['fold']}")
        pipeline = factory()
        result, topk_perda, topk_financeiro = evaluate_model_on_fold(
            model_name=model_name,
            pipeline=pipeline,
            df=df_dev,
            train_idx=fold["train_idx"],
            val_idx=fold["val_idx"],
            features=feature_list,
        )
        result["fold"] = fold["fold"]
        result["qtd_train"] = fold["qtd_train"]
        result["qtd_val"] = fold["qtd_val"]
        result["train_end_date"] = fold["train_end_date"]
        result["val_start_date"] = fold["val_start_date"]
        result["val_end_date"] = fold["val_end_date"]
        walk_forward_results.append(result)

        topk_perda["model"] = model_name
        topk_perda["fold"] = fold["fold"]
        topk_perda_results.append(topk_perda)

        if not topk_financeiro.empty:
            topk_financeiro["model"] = model_name
            topk_financeiro["fold"] = fold["fold"]
            topk_financeiro_results.append(topk_financeiro)

walk_forward_results = pd.DataFrame(walk_forward_results)
topk_perda_results = pd.concat(topk_perda_results, ignore_index=True)
topk_financeiro_results = pd.concat(topk_financeiro_results, ignore_index=True) if topk_financeiro_results else pd.DataFrame()

save_report(walk_forward_results, "36_3b_walk_forward_metrics.csv")
save_report(topk_perda_results, "37_3b_walk_forward_topk_risco_perda.csv")
if not topk_financeiro_results.empty:
    save_report(topk_financeiro_results, "38_3b_walk_forward_topk_prioridade_financeira.csv")

walk_forward_results.sort_values(["pr_auc_perda", "roc_auc_perda"], ascending=False)

## 13. Resumo por modelo

In [ ]:
model_summary = (
    walk_forward_results
    .groupby("model")
    .agg(
        mean_roc_auc_perda=("roc_auc_perda", "mean"),
        std_roc_auc_perda=("roc_auc_perda", "std"),
        mean_pr_auc_perda=("pr_auc_perda", "mean"),
        std_pr_auc_perda=("pr_auc_perda", "std"),
        mean_taxa_perda_val=("taxa_perda_val", "mean"),
        mean_best_f1_perda=("best_f1_perda", "mean"),
        mean_best_f1_threshold_perda=("best_f1_threshold_perda", "mean"),
        mean_precision_perda_t05=("precision_perda_t05", "mean"),
        mean_recall_perda_t05=("recall_perda_t05", "mean"),
        mean_f1_perda_t05=("f1_perda_t05", "mean"),
    )
    .reset_index()
    .sort_values("mean_pr_auc_perda", ascending=False)
)

save_report(model_summary, "39_3b_model_summary.csv")
model_summary

## 14. Top-k risco de perda por modelo

In [ ]:
topk_perda_summary = (
    topk_perda_results
    .groupby(["model", "top_k"])
    .agg(
        mean_precision_perda_at_k=("precision_perda_at_k", "mean"),
        mean_recall_perda_at_k=("recall_perda_at_k", "mean"),
        mean_lift_perda_at_k=("lift_perda_at_k", "mean"),
        mean_taxa_perda_base=("taxa_perda_base", "mean"),
    )
    .reset_index()
    .sort_values(["top_k", "mean_precision_perda_at_k"], ascending=[True, False])
)

save_report(topk_perda_summary, "40_3b_topk_risco_perda_summary.csv")
topk_perda_summary

## 15. Top-k prioridade financeira por modelo

In [ ]:
if not topk_financeiro_results.empty:
    topk_financeiro_summary = (
        topk_financeiro_results
        .groupby(["model", "top_k"])
        .agg(
            mean_precision_perda_at_k=("precision_perda_at_k", "mean"),
            mean_recall_perda_at_k=("recall_perda_at_k", "mean"),
            mean_lift_perda_at_k=("lift_perda_at_k", "mean"),
            mean_share_valor_ajuizado_total=("share_valor_ajuizado_total", "mean"),
            mean_share_valor_perdas_total=("share_valor_perdas_total", "mean"),
        )
        .reset_index()
        .sort_values(["top_k", "mean_share_valor_perdas_total"], ascending=[True, False])
    )
    save_report(topk_financeiro_summary, "41_3b_topk_prioridade_financeira_summary.csv")
    display(topk_financeiro_summary)
else:
    print("Top-k financeiro não calculado.")

## 16. Escolher melhor modelo 3B

In [ ]:
best_model_name = model_summary.iloc[0]["model"]
print("Melhor modelo 3B por PR-AUC de perda:", best_model_name)

## 17. Treinar melhor modelo no Dev completo e avaliar Holdout

In [ ]:
best_pipeline = selected_models[best_model_name]()

X_dev = df_dev[feature_list].copy()
y_dev = df_dev[TARGET_COL].copy()
X_holdout = df_holdout[feature_list].copy()
y_holdout = df_holdout[TARGET_COL].copy()

best_pipeline.fit(X_dev, y_dev)
proba_perda_holdout = best_pipeline.predict_proba(X_holdout)[:, 1]

holdout_roc_auc_perda = roc_auc_score(y_holdout, proba_perda_holdout)
holdout_pr_auc_perda = average_precision_score(y_holdout, proba_perda_holdout)

best_thr, best_p, best_r, best_f1 = find_best_f1_threshold_perda(y_holdout, proba_perda_holdout)
thr_05 = threshold_metrics_perda(y_holdout, proba_perda_holdout, threshold=0.5)

holdout_metrics = pd.DataFrame([{
    "model": best_model_name,
    "roc_auc_perda": holdout_roc_auc_perda,
    "pr_auc_perda": holdout_pr_auc_perda,
    "taxa_perda_holdout": y_holdout.mean(),
    "taxa_ganho_holdout": 1 - y_holdout.mean(),
    "best_f1_threshold_perda": best_thr,
    "best_f1_precision_perda": best_p,
    "best_f1_recall_perda": best_r,
    "best_f1_perda": best_f1,
    "precision_perda_t05": thr_05["precision_perda"],
    "recall_perda_t05": thr_05["recall_perda"],
    "f1_perda_t05": thr_05["f1_perda"],
    "pred_perda_rate_t05": thr_05["pred_perda_rate"],
    "qtd_dev": len(df_dev),
    "qtd_holdout": len(df_holdout),
    "qtd_features": len(feature_list),
}])

save_report(holdout_metrics, "42_3b_holdout_best_model_metrics.csv")
holdout_metrics

## 18. Holdout top-k risco de perda

In [ ]:
holdout_topk_perda = topk_risco_perda_metrics(
    y_true=y_holdout.values,
    proba_perda=proba_perda_holdout,
    ks=(0.01, 0.05, 0.10, 0.20)
)
holdout_topk_perda["model"] = best_model_name
save_report(holdout_topk_perda, "43_3b_holdout_topk_risco_perda.csv")
holdout_topk_perda

## 19. Holdout top-k prioridade financeira

In [ ]:
if VALUE_COL in df_holdout.columns:
    holdout_topk_financeiro = topk_prioridade_financeira_metrics(
        y_true=y_holdout.values,
        proba_perda=proba_perda_holdout,
        valor_ajuizado=df_holdout[VALUE_COL],
        ks=(0.01, 0.05, 0.10, 0.20)
    )
    holdout_topk_financeiro["model"] = best_model_name
    save_report(holdout_topk_financeiro, "44_3b_holdout_topk_prioridade_financeira.csv")
    display(holdout_topk_financeiro)
else:
    print(f"{VALUE_COL} não encontrado.")

## 20. Ranking do holdout

In [ ]:
ranking_cols = ["Numerodistribuicao", "Identificador", DATE_COL, TARGET_COL_LEGACY, TARGET_COL, VALUE_COL]
ranking_cols = [c for c in ranking_cols if c in df_holdout.columns]

ranking_holdout = df_holdout[ranking_cols].copy()
ranking_holdout["target_classe_real"] = ranking_holdout[TARGET_COL].map({0: "banco_ganhou", 1: "banco_perdeu"})
ranking_holdout["proba_banco_perdeu"] = proba_perda_holdout
ranking_holdout["score_risco_perda"] = proba_perda_holdout

if VALUE_COL in ranking_holdout.columns:
    ranking_holdout["score_prioridade_financeira"] = (
        ranking_holdout["score_risco_perda"] *
        pd.to_numeric(ranking_holdout[VALUE_COL], errors="coerce").fillna(0)
    )
else:
    ranking_holdout["score_prioridade_financeira"] = np.nan

ranking_holdout["rank_risco_perda"] = ranking_holdout["score_risco_perda"].rank(ascending=False, method="first").astype(int)
ranking_holdout["rank_prioridade_financeira"] = ranking_holdout["score_prioridade_financeira"].rank(ascending=False, method="first").astype(int)

ranking_risco = ranking_holdout.sort_values("score_risco_perda", ascending=False)
ranking_financeiro = ranking_holdout.sort_values("score_prioridade_financeira", ascending=False)

ranking_risco_path = PROCESSED_DIR / "ranking_holdout_risco_perda_3b_encoder_model.parquet"
ranking_financeiro_path = PROCESSED_DIR / "ranking_holdout_prioridade_financeira_3b_encoder_model.parquet"

ranking_risco.to_parquet(ranking_risco_path, index=False)
ranking_financeiro.to_parquet(ranking_financeiro_path, index=False)

save_report(ranking_risco.head(1000), "45_3b_ranking_holdout_top1000_risco_perda.csv")
save_report(ranking_financeiro.head(1000), "46_3b_ranking_holdout_top1000_prioridade_financeira.csv")

ranking_risco.head(20)

## 21. Classification report

In [ ]:
pred_perda_holdout_t05 = (proba_perda_holdout >= 0.5).astype(int)

print("Classification report — threshold 0.5")
print("Classe 0 = banco ganhou")
print("Classe 1 = banco perdeu")
print(classification_report(
    y_holdout,
    pred_perda_holdout_t05,
    target_names=["banco_ganhou", "banco_perdeu"]
))

print("Confusion matrix — threshold 0.5")
print("Linhas = real | Colunas = previsto")
print(confusion_matrix(y_holdout, pred_perda_holdout_t05))

## 22. Comparação com baseline 3A

In [ ]:
baseline_3a = {
    "model": "baseline_3a_random_forest_onehot_tfidf",
    "holdout_pr_auc_perda": 0.466804,
    "holdout_roc_auc_perda": 0.77858,
    "top5_precision_perda": 0.603004,
    "top10_fin_share_valor_perdas": 0.502560,
}

current_top5 = holdout_topk_perda.loc[holdout_topk_perda["top_k"] == 0.05, "precision_perda_at_k"].iloc[0]

if VALUE_COL in df_holdout.columns:
    current_top10_fin = holdout_topk_financeiro.loc[
        holdout_topk_financeiro["top_k"] == 0.10,
        "share_valor_perdas_total"
    ].iloc[0]
else:
    current_top10_fin = np.nan

comparison_3a_3b = pd.DataFrame([
    baseline_3a,
    {
        "model": f"3b_{best_model_name}",
        "holdout_pr_auc_perda": holdout_pr_auc_perda,
        "holdout_roc_auc_perda": holdout_roc_auc_perda,
        "top5_precision_perda": current_top5,
        "top10_fin_share_valor_perdas": current_top10_fin,
    }
])

comparison_3a_3b["delta_pr_auc_vs_3a"] = comparison_3a_3b["holdout_pr_auc_perda"] - comparison_3a_3b.loc[0, "holdout_pr_auc_perda"]
comparison_3a_3b["delta_roc_auc_vs_3a"] = comparison_3a_3b["holdout_roc_auc_perda"] - comparison_3a_3b.loc[0, "holdout_roc_auc_perda"]
comparison_3a_3b["delta_top5_precision_vs_3a"] = comparison_3a_3b["top5_precision_perda"] - comparison_3a_3b.loc[0, "top5_precision_perda"]
comparison_3a_3b["delta_top10_fin_vs_3a"] = comparison_3a_3b["top10_fin_share_valor_perdas"] - comparison_3a_3b.loc[0, "top10_fin_share_valor_perdas"]

save_report(comparison_3a_3b, "47_3b_comparison_vs_baseline_3a.csv")
comparison_3a_3b

## 23. Summary final

In [ ]:
summary_step_3b = pd.DataFrame([{
    "target_semantica": "0=banco_ganhou | 1=banco_perdeu",
    "run_mode": RUN_MODE,
    "best_model_3b_walk_forward": best_model_name,
    "best_model_3b_mean_pr_auc_perda_wf": model_summary.loc[model_summary["model"] == best_model_name, "mean_pr_auc_perda"].iloc[0],
    "best_model_3b_mean_roc_auc_perda_wf": model_summary.loc[model_summary["model"] == best_model_name, "mean_roc_auc_perda"].iloc[0],
    "holdout_pr_auc_perda": holdout_pr_auc_perda,
    "holdout_roc_auc_perda": holdout_roc_auc_perda,
    "taxa_perda_holdout": y_holdout.mean(),
    "taxa_ganho_holdout": 1 - y_holdout.mean(),
    "qtd_features": len(feature_list),
    "qtd_dev": len(df_dev),
    "qtd_holdout": len(df_holdout),
    "ranking_risco_holdout_path": str(ranking_risco_path),
    "ranking_financeiro_holdout_path": str(ranking_financeiro_path),
}])

save_report(summary_step_3b, "48_summary_step_3b_encoders_feature_engine.csv")
summary_step_3b.T

# O que verificar após rodar

Enviar na próxima iteração:

1. `model_summary`
2. `holdout_metrics`
3. `holdout_topk_perda`
4. `holdout_topk_financeiro`
5. `comparison_3a_3b`
6. `summary_step_3b.T`

## Como decidir se a etapa 3B foi melhor que 3A

O modelo 3B só deve substituir o baseline 3A se melhorar pelo menos uma destas frentes sem piorar muito as demais:

1. PR-AUC perda no holdout;
2. top 5% por risco de perda;
3. top 10% por prioridade financeira;
4. estabilidade no walk-forward.

## Próxima etapa

Se 3B melhorar o baseline:

```text
Etapa 3C — CatBoost nativo com categóricas e texto
```

Se 3B não melhorar:

```text
Manter 3A como baseline forte e seguir para CatBoost mesmo assim.
```